# DVC Practical Quiz — Data Version Control using DVC
**Name:** Student

**Instructions:** Dummy data is used. All cells are executable in order.

---
## Section A — Repository Setup
### Q1. Create `dvc_lab_quiz`, initialize Git and DVC

In [1]:
import os, shutil, sys, stat

def rm_readonly(func, path, _):
    os.chmod(path, __import__('stat').S_IWRITE)
    func(path)

base = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'dvc_lab_quiz' else os.getcwd()
target = os.path.join(base, 'dvc_lab_quiz')
if os.path.exists(target):
    shutil.rmtree(target, onerror=rm_readonly)
os.makedirs(target)
os.chdir(target)
print('Python:', sys.executable)
print('Working directory:', os.getcwd())

Python: C:\Users\HP\AppData\Local\Programs\Python\Python314\python.exe
Working directory: D:\semister 8\ML_OPS\DVC online quiz\dvc_lab_quiz


In [2]:
!git init
!git config user.email "student@example.com"
!git config user.name "Student"

Initialized empty Git repository in D:/semister 8/ML_OPS/DVC online quiz/dvc_lab_quiz/.git/


In [3]:
import sys
!{sys.executable} -m dvc init
!git add .
!git commit -m "Initialize DVC"

Initialized DVC repository.

You can now commit the changes to git.

+---------------------------------------------------------------------+
|                                                                     |
|        DVC has enabled anonymous aggregate usage analytics.         |
|     Read the analytics documentation (and how to opt-out) here:     |
|             <https://dvc.org/doc/user-guide/analytics>              |
|                                                                     |
+---------------------------------------------------------------------+

What's next?
------------
- Check out the documentation: <https://dvc.org/doc>
- Get help and share ideas: <https://dvc.org/chat>
- Star us on GitHub: <https://github.com/treeverse/dvc>


[master (root-commit) 7c3fcf7] Initialize DVC
 3 files changed, 6 insertions(+)
 create mode 100644 .dvc/.gitignore
 create mode 100644 .dvc/config
 create mode 100644 .dvcignore


### Commands Used:
```bash
mkdir dvc_lab_quiz
cd dvc_lab_quiz
git init
dvc init
```

### Files/Folders DVC Generates:
| File/Folder | Description |
|---|---|
| `.dvc/` | Main DVC config directory |
| `.dvc/config` | DVC project configuration |
| `.dvc/cache/` | Local cache for data file content |
| `.dvc/.gitignore` | Tells Git to ignore DVC cache |
| `.dvcignore` | Like `.gitignore` but for DVC |

**Explanation:**
- `git init` creates `.git/` — tracks code and metadata.
- `dvc init` creates `.dvc/` — manages large data files outside Git.

---
## Section B — Adding and Tracking Data
### Q2. Create `data/sample.csv` and track with DVC

In [4]:
import os
os.makedirs('data', exist_ok=True)
csv_content = 'id,name,score\n1,Alice,85\n2,Bob,90\n3,Charlie,78\n4,Diana,92\n5,Eve,88\n'
with open('data/sample.csv', 'w') as f:
    f.write(csv_content)
print('data/sample.csv created:')
print(csv_content)

data/sample.csv created:
id,name,score
1,Alice,85
2,Bob,90
3,Charlie,78
4,Diana,92
5,Eve,88



In [5]:
import sys
!{sys.executable} -m dvc add data/sample.csv


To track the changes with git, run:

	git add 'data\.gitignore' 'data\sample.csv.dvc'

To enable auto staging, run:

	dvc config core.autostage true


\u280b Checking graph



In [6]:
# View the generated .dvc pointer file
with open('data/sample.csv.dvc') as f:
    print(f.read())

outs:
- md5: b63e06e7cd1a2de0bf8d1607b6d9ff50
  size: 73
  hash: md5
  path: sample.csv



In [7]:
!git add data/sample.csv.dvc data/.gitignore
!git commit -m "Track sample.csv with DVC"

[master 60b7001] Track sample.csv with DVC
 2 files changed, 6 insertions(+)
 create mode 100644 data/.gitignore
 create mode 100644 data/sample.csv.dvc


### Command Used:
```bash
dvc add data/sample.csv
```

### The `.dvc` File Created: `data/sample.csv.dvc`
```yaml
outs:
- md5: a1b2c3d4...   # MD5 hash of file content
  size: 72           # File size in bytes
  path: sample.csv   # Relative path
```

**Explanation:**
- The actual data goes into `.dvc/cache/` (keyed by MD5 hash).
- The `.dvc` file is a small pointer that Git tracks instead of the large file.
- `data/.gitignore` is updated to exclude `sample.csv` from Git.

---
## Section C — Remote Storage
### Q3. Simulate DVC remote storage

In [8]:
import os, sys
remote_path = os.path.abspath('../remote_storage').replace('\\', '/')
os.makedirs(remote_path, exist_ok=True)
import subprocess
result = subprocess.run(
    [sys.executable, '-m', 'dvc', 'remote', 'add', '-d', 'myremote', remote_path],
    capture_output=True, text=True
)
print(result.stdout)
if result.stderr: print('INFO:', result.stderr[:200])
result2 = subprocess.run([sys.executable, '-m', 'dvc', 'remote', 'list'], capture_output=True, text=True)
print(result2.stdout)

Setting 'myremote' as a default remote.



myremote        D:/semister 8/ML_OPS/DVC online quiz/remote_storage     
(default)



In [9]:
!git add .dvc/config
!git commit -m "Add DVC remote storage"

[master ed085ce] Add DVC remote storage
 1 file changed, 4 insertions(+)


### Commands Used:
```bash
dvc remote add -d myremote /path/to/remote_storage
# For cloud:
# dvc remote add -d myremote s3://my-bucket/dvc-storage
dvc remote list
```

### What the Remote Does:
| Role | Description |
|---|---|
| **Central Storage** | Stores large data/model files outside Git |
| **Team Sharing** | Teammates use `dvc pull` to get exact data |
| **Versioned Backup** | Every data version is retrievable by hash |
| **Decoupled** | Data lives separately from code repo |

- `-d` sets this as the **default** remote (saved in `.dvc/config`).
- Supports: local folders, AWS S3, GCS, Azure Blob, SSH, HDFS.

---
## Section D — Creating a Simple Pipeline
### Q4. Define `process_data.py` as a DVC pipeline stage

In [10]:
script = 'import csv\n\nwith open("data/sample.csv", "r") as f:\n    reader = csv.DictReader(f)\n    rows = list(reader)\n\nprocessed = [r for r in rows if int(r["score"]) > 80]\n\nwith open("processed.csv", "w", newline="") as f:\n    writer = csv.DictWriter(f, fieldnames=["id","name","score"])\n    writer.writeheader()\n    writer.writerows(processed)\n\nprint(f"Processed {len(processed)} of {len(rows)} rows")\n'
with open('process_data.py', 'w') as f:
    f.write(script)
print('process_data.py created')

process_data.py created


In [11]:
import sys
PYTHON = sys.executable
!{PYTHON} -m dvc stage add -n process_data -d data/sample.csv -d process_data.py -o processed.csv "{PYTHON} process_data.py"

Added stage 'process_data' in 'dvc.yaml'

To track the changes with git, run:

	git add dvc.yaml .gitignore

To enable auto staging, run:

	dvc config core.autostage true


In [12]:
import sys
!{sys.executable} -m dvc repro

'data\sample.csv.dvc' didn't change, skipping
Running stage 'process_data':
> C:\Users\HP\AppData\Local\Programs\Python\Python314\python.exe process_data.py
Processed 4 of 5 rows
Generating lock file 'dvc.lock'
Updating lock file 'dvc.lock'

To track the changes with git, run:

	git add dvc.lock

To enable auto staging, run:

	dvc config core.autostage true
Use `dvc push` to send your updates to remote storage.


In [13]:
# Show the generated dvc.yaml
with open('dvc.yaml') as f:
    print(f.read())

stages:
  process_data:
    cmd: C:\Users\HP\AppData\Local\Programs\Python\Python314\python.exe 
      process_data.py
    deps:
    - data/sample.csv
    - process_data.py
    outs:
    - processed.csv



In [14]:
!git add dvc.yaml dvc.lock .gitignore process_data.py
!git commit -m "Add process_data pipeline stage"

[master 0df3d41] Add process_data pipeline stage
 4 files changed, 43 insertions(+)
 create mode 100644 .gitignore
 create mode 100644 dvc.lock
 create mode 100644 dvc.yaml
 create mode 100644 process_data.py


### Command Used:
```bash
dvc stage add -n process_data \
    -d data/sample.csv \
    -d process_data.py \
    -o processed.csv \
    python process_data.py

dvc repro
```

### Flag Explanation:
| Flag | Meaning |
|---|---|
| `-n process_data` | Stage name |
| `-d data/sample.csv` | Input dependency |
| `-d process_data.py` | Script dependency |
| `-o processed.csv` | Output file |

### Files DVC Uses for Pipeline:
- **`dvc.yaml`** — defines all stages, deps, and outputs.
- **`dvc.lock`** — records exact MD5 hashes after each run (enables reproducibility).

---
## Section E — Tracking and Comparing Metrics
### Q5. Track and compare metrics across versions

In [15]:
import json
metrics = {"accuracy": 0.82, "precision": 0.80, "recall": 0.85, "f1_score": 0.82}
with open('metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print('metrics.json (version 1):')
print(json.dumps(metrics, indent=2))

metrics.json (version 1):
{
  "accuracy": 0.82,
  "precision": 0.8,
  "recall": 0.85,
  "f1_score": 0.82
}


In [16]:
!git add metrics.json
!git commit -m "Add metrics v1 accuracy=0.82"

[master f5869d4] Add metrics v1 accuracy=0.82
 1 file changed, 6 insertions(+)
 create mode 100644 metrics.json


In [17]:
import sys
!{sys.executable} -m dvc metrics show metrics.json

Path          accuracy    f1_score    precision    recall
metrics.json  0.82        0.82        0.8          0.85


In [18]:
import json
metrics_v2 = {"accuracy": 0.87, "precision": 0.85, "recall": 0.89, "f1_score": 0.87}
with open('metrics.json', 'w') as f:
    json.dump(metrics_v2, f, indent=2)
print('metrics.json (version 2):')
print(json.dumps(metrics_v2, indent=2))
!git add metrics.json
!git commit -m "Update metrics v2 accuracy=0.87"

metrics.json (version 2):
{
  "accuracy": 0.87,
  "precision": 0.85,
  "recall": 0.89,
  "f1_score": 0.87
}


[master f7f963a] Update metrics v2 accuracy=0.87
 1 file changed, 4 insertions(+), 4 deletions(-)


In [19]:
import sys
!{sys.executable} -m dvc metrics diff HEAD~1 HEAD


### Commands Used:
```bash
dvc metrics show metrics.json
dvc metrics diff HEAD~1 HEAD
```

### How DVC Tracks Metrics Across Versions:
1. Metrics file is committed to Git at each experiment.
2. `dvc metrics show` reads values from the current checkout.
3. `dvc metrics diff` compares between any two Git commits/branches.
4. No manual spreadsheets — DVC handles all version comparison automatically.

---
## Section F — Reproduction and Version Control
### Q6. Rerun pipeline automatically after dataset change

In [20]:
import sys
with open('data/sample.csv', 'a') as f:
    f.write('6,Frank,95\n7,Grace,73\n8,Heidi,88\n')
print('Dataset updated with 3 new rows')
!{sys.executable} -m dvc add data/sample.csv
!git add data/sample.csv.dvc
!git commit -m "Update dataset with new rows"

Dataset updated with 3 new rows



To track the changes with git, run:

	git add 'data\sample.csv.dvc'

To enable auto staging, run:

	dvc config core.autostage true


\u280b Checking graph



[master 63c0cd1] Update dataset with new rows


 1 file changed, 2 insertions(+), 2 deletions(-)


In [21]:
import sys
!{sys.executable} -m dvc repro

'data\sample.csv.dvc' didn't change, skipping
Running stage 'process_data':
> C:\Users\HP\AppData\Local\Programs\Python\Python314\python.exe process_data.py
Processed 6 of 8 rows
Updating lock file 'dvc.lock'

To track the changes with git, run:

	git add dvc.lock

To enable auto staging, run:

	dvc config core.autostage true
Use `dvc push` to send your updates to remote storage.


In [22]:
# Verify output changed
with open('processed.csv') as f:
    print(f.read())
!git add dvc.lock
!git commit -m "Reproduced pipeline after dataset update"

id,name,score
1,Alice,85
2,Bob,90
4,Diana,92
5,Eve,88
6,Frank,95
8,Heidi,88



[master 8c18595] Reproduced pipeline after dataset update
 1 file changed, 4 insertions(+), 4 deletions(-)


### Command Used:
```bash
dvc repro
```

### How DVC Decides Which Stages to Rerun:
| Condition | DVC Action |
|---|---|
| Dependency hash changed | Reruns that stage + all downstream |
| Script changed | Reruns affected stage |
| Output hash matches lock | Skips (uses cache) |
| Nothing changed | Skips all — 'Pipeline is up to date' |

DVC builds a **DAG** from `dvc.yaml`, computes MD5 hashes of all current files, compares against `dvc.lock`, and only reruns what changed.

---
## Section G — Final Versioning
### Q7. Push all updates to version control

In [23]:
import sys, subprocess
result = subprocess.run([sys.executable, '-m', 'dvc', 'push'], capture_output=True, text=True, cwd=os.getcwd())
print(result.stdout if result.stdout else 'Push complete')
if result.returncode != 0:
    print('Note:', result.stderr[:300])
else:
    print('Data pushed to DVC remote successfully!')


2 files pushed

Data pushed to DVC remote successfully!


In [24]:
# Verify remote storage received the files
import os
remote_path = os.path.abspath('../remote_storage')
files = []
for dp, dn, fn in os.walk(remote_path):
    for f in fn:
        files.append(os.path.join(dp, f))
print(f'Files stored in DVC remote ({len(files)} total):')
for f in files:
    print(' ', f)

Files stored in DVC remote (2 total):
  D:\semister 8\ML_OPS\DVC online quiz\remote_storage\files\md5\27\a33495a1c74f6ed042754fbb61cb64
  D:\semister 8\ML_OPS\DVC online quiz\remote_storage\files\md5\78\e25bea2d672e40c2d760bbcd0d4983


In [25]:
print('Final Git log:')
!git log --oneline

Final Git log:


8c18595 Reproduced pipeline after dataset update
63c0cd1 Update dataset with new rows
f7f963a Update metrics v2 accuracy=0.87
f5869d4 Add metrics v1 accuracy=0.82
0df3d41 Add process_data pipeline stage
ed085ce Add DVC remote storage
60b7001 Track sample.csv with DVC
7c3fcf7 Initialize DVC


### Commands Used:
```bash
# Git: push code + .dvc pointer files
git add .
git commit -m "Final version"
git push origin main

# DVC: push actual data to remote
dvc push
```

### How a Teammate Reproduces Results:
```bash
git clone https://github.com/username/dvc_lab_quiz.git
cd dvc_lab_quiz
dvc pull        # Downloads exact data versions
dvc repro       # Reruns pipeline
dvc metrics show
```

### Why Exact Reproduction is Guaranteed:
| Component | Role |
|---|---|
| **Git** | Locks code, `dvc.yaml`, `dvc.lock` versions |
| **`dvc.lock`** | Records MD5 hash of every input/output |
| **DVC Remote** | Stores all data versions by content hash |
| **`dvc pull`** | Fetches data matching the locked hashes |
| **`dvc repro`** | Re-executes stages with identical inputs |

**Key insight:** Git commit + `dvc.lock` = complete snapshot of code AND data → identical results on any machine.

---
## Summary Table
| Section | Key Command | Purpose |
|---|---|---|
| A | `git init` + `dvc init` | Set up Git and DVC |
| B | `dvc add data/sample.csv` | Track data with DVC |
| C | `dvc remote add -d myremote <path>` | Connect remote storage |
| D | `dvc stage add -n ... && dvc repro` | Define and run pipeline |
| E | `dvc metrics show/diff` | View and compare metrics |
| F | `dvc repro` | Smart pipeline reproduction |
| G | `git push` + `dvc push` | Push code and data |